# Gold 03 — Customer sales

**Grain:** one row per customer. All Silver customers remain present, even those with no completed order. Only completed orders count toward revenue, order count, average order value, and last order date. Revenue is subtotal minus discount, excluding tax.


In [ ]:
import os  # Set Java before Spark starts.
import shutil  # Find Linux Java.
from pathlib import Path  # Build data paths.
from pyspark.sql import SparkSession, functions as F  # Use Spark and column expressions.
from delta import configure_spark_with_delta_pip  # Add Delta Lake to Spark.
java = shutil.which("java")  # Locate WSL Java.
if java is None:  # Require Java.
    raise RuntimeError("Java is unavailable in WSL")  # Explain a startup failure.
os.environ["JAVA_HOME"] = str(Path(java).resolve().parent.parent)  # Avoid the Windows Java path.
project_root = Path("/mnt/f/Big Data/Retail_Lakehouse_Learning")  # Find the project in WSL.
if not project_root.is_dir():  # Check the mounted drive.
    raise FileNotFoundError(project_root)  # Stop on an incorrect path.
builder = (SparkSession.builder.master("local[2]")  # Use two local threads.
    .appName("RetailGoldDailyStoreSales")  # Label this job.
    .config("spark.sql.shuffle.partitions", "4")  # Keep small local shuffles manageable.
    .config("spark.ui.enabled", "false")  # Avoid a second Spark UI port.
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Enable Delta SQL.
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))  # Register Delta.
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # Start or reuse Spark.
print("Spark:", spark.version, "Timezone:", spark.conf.get("spark.sql.session.timeZone"))  # Record session behavior.


In [ ]:
silver_root = project_root / "lakehouse" / "silver" / "data"  # Locate cleaned tables.
gold_root = project_root / "lakehouse" / "gold" / "data"  # Locate Gold tables.
customers_df = spark.read.format("delta").load(str(silver_root / "customers"))  # Read the full customer dimension.
orders_df = spark.read.format("delta").load(str(silver_root / "orders"))  # Read typed orders.
daily_df = spark.read.format("delta").load(str(gold_root / "daily_store_sales"))  # Read checked sales total.
if customers_df.count() != customers_df.select("customer_id").distinct().count():  # Require one row per customer.
    raise ValueError("Duplicate Silver customer keys")  # Prevent doubled customer totals.
completed_df = (orders_df.filter(F.col("status") == "completed")  # Keep actual sales.
    .withColumn("revenue", (F.col("subtotal") - F.col("discount_amount")).cast("decimal(18,2)"))  # Calculate net revenue.
    .withColumn("date", F.to_date("order_ts")))  # Keep the order date.
if completed_df.filter(F.col("customer_id").isNull() | F.col("date").isNull() | F.col("revenue").isNull()).count():  # Require group keys and money.
    raise ValueError("Invalid completed order for customer sales")  # Stop incomplete results.
print("Customers:", customers_df.count(), "Completed orders:", completed_df.count())  # Show inputs.


In [ ]:
customer_facts_df = completed_df.groupBy("customer_id").agg(  # Reduce completed orders to one row per customer.
    F.count("*").alias("orders"),  # Count sales orders.
    F.sum("revenue").cast("decimal(18,2)").alias("revenue"),  # Sum net sales.
    F.max("date").alias("last_order_date"))  # Find latest completed order date.
customer_gold_df = (customers_df  # Keep all customers, including zero-sales customers.
    .join(customer_facts_df, "customer_id", "left")  # Attach completed-order totals.
    .withColumn("orders", F.coalesce(F.col("orders"), F.lit(0)))  # Represent missing sales with zero.
    .withColumn("revenue", F.coalesce(F.col("revenue"), F.lit(0).cast("decimal(18,2)")))  # Keep a typed zero.
    .withColumn("avg_order_value", F.when(F.col("orders") > 0, F.bround(F.col("revenue") / F.col("orders"), 2)).otherwise(F.lit(0)).cast("decimal(18,2)"))  # Divide only for paying customers.
    .select("customer_id", "first_name", "last_name", "city", "loyalty_tier", "active_flag", "orders", "revenue", "avg_order_value", "last_order_date"))  # Exclude contact details from reporting.
if customer_gold_df.count() != customers_df.count():  # Check the left join preserved all customers.
    raise ValueError("Customer Gold row count changed")  # Catch multiplication or loss.
if customer_gold_df.count() != customer_gold_df.select("customer_id").distinct().count():  # Check Gold grain.
    raise ValueError("Duplicate customer Gold keys")  # Prevent repeated customers.
daily_revenue = daily_df.agg(F.sum("revenue").alias("v")).first().v  # Get the validated daily total.
customer_revenue = customer_gold_df.agg(F.sum("revenue").alias("v")).first().v  # Recompute by customer.
daily_orders = daily_df.agg(F.sum("orders").alias("v")).first().v  # Get validated order count.
customer_orders = customer_gold_df.agg(F.sum("orders").alias("v")).first().v  # Recompute by customer.
if customer_revenue != daily_revenue or customer_orders != daily_orders:  # Require both Gold views to agree.
    raise ValueError("Customer totals differ from daily store sales")  # Stop financial inconsistency.
print("Customer rows:", customer_gold_df.count(), "Revenue:", customer_revenue, "Orders:", customer_orders)  # Show checks.
print("Zero-sale customers:", customer_gold_df.filter(F.col("orders") == 0).count())  # Show retained customers.
customer_gold_df.orderBy("customer_id").show(5, truncate=False)  # Inspect sample rows.


In [ ]:
gold_path = gold_root / "customer_sales"  # Choose Gold Delta location.
gold_path.parent.mkdir(parents=True, exist_ok=True)  # Create the ignored data parent.
customer_gold_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(str(gold_path))  # Commit the Gold snapshot.
saved_df = spark.read.format("delta").load(str(gold_path))  # Reopen saved data.
if saved_df.count() != customers_df.count():  # Verify persisted grain.
    raise ValueError("Saved customer row count differs")  # Surface write issues.
if saved_df.agg(F.sum("revenue").alias("v")).first().v != daily_revenue:  # Verify persisted money.
    raise ValueError("Saved customer revenue differs")  # Surface financial issues.
print("Saved customer_sales:", saved_df.count(), "Delta log:", (gold_path / "_delta_log").is_dir())  # Confirm Delta commit.
saved_df.printSchema()  # Show reporting types.
